# LLM07: FlashAttention — GPU-Efficient Attention

## Lab Overview

Standard self-attention has $O(N^2)$ HBM (High-Bandwidth Memory) access, making it the main bottleneck for long sequences. **FlashAttention** reduces memory access by leveraging GPU memory hierarchy: keeping intermediate results in fast on-chip SRAM (shared memory / registers) and minimizing round-trips to slow off-chip HBM (global memory).

This lab walks through the GPU memory model, tiling, online softmax, and the FlashAttention algorithm, with runnable code so you can benchmark standard vs. flash attention yourself.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

Understand why FlashAttention is faster (I/O analysis, not just FLOPs), and see the speedup on real hardware.

1. **Describe the GPU memory hierarchy**: registers → shared memory (SRAM) → L2 cache → HBM (DRAM).
2. **Understand Tiling for MatMul**: How tile-based computation reduces global memory access.
3. **Implement Online Softmax**: Numerically stable, incremental softmax that avoids materializing the full $N \times N$ attention matrix.
4. **Grasp FlashAttention V1 & V2**: The key algorithmic improvements and their IO complexity.
5. **Benchmark**: Compare standard attention vs. `torch.nn.functional.scaled_dot_product_attention` (which uses FlashAttention internally).

---


## 1. Environment Setup

In [2]:
import math
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory (HBM): {props.total_memory / 1e9:.1f} GB")
    print(f"SM count: {props.multi_processor_count}")

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.10.0+rocm7.1
GPU: Radeon RX 7900 XTX
GPU Memory (HBM): 25.8 GB
SM count: 48


## 2. GPU Memory Hierarchy — Why IO Matters

| Level | Name | Typical Size (A100) | Bandwidth | Latency |
|-------|------|--------------------:|----------:|--------:|
| Registers | per-thread storage | ~256 KB / SM | — | ~1 cycle |
| Shared Memory (SRAM) | on-chip, per-SM | 164 KB / SM (configurable) | ~19 TB/s | ~30 cycles |
| L2 Cache | on-chip, shared | 40 MB | ~5 TB/s | ~200 cycles |
| HBM (Global Memory) | off-chip DRAM | 80 GB | 2.0 TB/s | ~400+ cycles |

**Key insight**: Shared memory is **~10×** faster than HBM. If we can keep intermediate results (attention scores, softmax statistics) in SRAM instead of writing them back to HBM, we save massive amounts of IO.



### Measuring Actual Memory Bandwidth on This Machine

Let's measure the actual memory bandwidth by performing simple memory-bound operations. We'll compare:
1. **HBM bandwidth**: Large tensor copy (global memory access)
2. **SRAM bandwidth**: Small tensor operations that fit in cache/shared memory

**Note**: This is a simplified benchmark. Real GPU kernels can achieve higher bandwidth with optimized memory access patterns.

In [13]:

def measure_memory_bandwidth(device, size_mb=256):
    """Measure approximate HBM bandwidth using tensor copy.

    Args:
        device: torch device
        size_mb: size of tensor in MB

    Returns:
        bandwidth_gb_s: measured bandwidth in GB/s
    """
    # Hint: size_mb * 1e6 / 4 elements for float32
    num_elements = int(size_mb * 1e6 / 4)

    # Create source tensor
    src = torch.randn(num_elements, device=device, dtype=torch.float32)

    # Warmup
    for _ in range(3):
        _ = src.clone()
        if device.type == "cuda":
            torch.cuda.synchronize()

    # Benchmark
    repeats = 10
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        _ = src.clone()
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / repeats

    # TODO: Calculate bandwidth
    # Hint: bytes transferred / time, convert to GB/s
    bytes_transferred = num_elements * 4  # 4 bytes per float32
    bandwidth_gb_s = (bytes_transferred / elapsed) / 1e9

    return bandwidth_gb_s


if device.type == "cuda":
    print("=== Memory Bandwidth Measurement ===")

    # Measure HBM bandwidth (large tensor, doesn't fit in cache)
    hbm_bw = measure_memory_bandwidth(device, size_mb=256)
    print(f"HBM Bandwidth (estimated): {hbm_bw:.1f} GB/s")

    # Get device properties
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"SM Count: {props.multi_processor_count}")

    # Hint: Compare measured HBM bandwidth with typical SRAM bandwidth (~19 TB/s)
    sram_bw_estimate = 19000  # GB/s (typical for modern GPUs)
    bandwidth_ratio = sram_bw_estimate / hbm_bw
    print(f"SRAM is approximately {bandwidth_ratio:.0f}× faster than HBM")
else:
    print("GPU not available — bandwidth measurement skipped.")

=== Memory Bandwidth Measurement ===
HBM Bandwidth (estimated): 391.3 GB/s
GPU: Radeon RX 7900 XTX
SM Count: 48
SRAM is approximately 49× faster than HBM


### Standard Attention IO Analysis

For a sequence of length $N$ and head dimension $d$:

1. Read Q, K from HBM → compute $S = QK^T$ → write $S$ to HBM  
2. Read $S$ → compute $P = \text{softmax}(S)$ → write $P$ to HBM  
3. Read $P$, V → compute $O = PV$ → write $O$ to HBM  

**Total HBM access**: $O(Nd + N^2)$ — dominated by the $N \times N$ matrices $S$ and $P$.

In [ ]:
# TODO: Implement standard attention
# Standard attention — materializes the full N×N score matrix
def standard_attention(Q, K, V):
    """Naive scaled dot-product attention.
    Args: Q, K, V — (batch, num_heads, seq_len, head_dim)
    Returns: output — same shape as V

    Steps:
    1. Compute attention scores: Q @ K^T / sqrt(d_k)
    2. Apply softmax to get attention weights
    3. Apply attention weights to values
    """
    # Get head dimension
    d_k = Q.size(-1)

    # TODO: Compute scaled attention scores
    # Hint: Use torch.matmul and transpose the last two dimensions of K
    scores =  torch.matmul(None) / math.sqrt(d_k) # Replace with correct code

    # TODO: Apply softmax along the last dimension
    attn_weights = torch.softmax(None, dim=-1)  # Replace with correct code

    # TODO: Apply attention weights to values
    # Hint: Use torch.matmul to multiply attn_weights with V
    output = torch.matmul(None, None)  # Replace with correct code

    return output

# Quick shape check
B, H, N, D = 1, 4, 8, 16
Q = torch.randn(B, H, N, D, device=device)
K = torch.randn(B, H, N, D, device=device)
V = torch.randn(B, H, N, D, device=device)
out = standard_attention(Q, K, V)
print(f"Standard attention output shape: {out.shape}")
print(f"Score matrix size: ({N}×{N}) = {N*N} elements per head")
print(f"  → For N=4096: {4096*4096:,} elements = {4096*4096*2/1e6:.1f} MB (FP16) per head")

Standard attention output shape: torch.Size([1, 4, 8, 16])
Score matrix size: (8×8) = 64 elements per head
  → For N=4096: 16,777,216 elements = 33.6 MB (FP16) per head


## 3. Tiling for Matrix Multiplication

**Tiling** partitions large matrices into blocks (tiles) that fit in shared memory. Each tile is loaded once and reused for multiple computations, reducing global memory reads.

### Example: 32×32 MatMul with 16×16 tiles
- Without tiling: each element of C reads 32 values of A and 32 values of B from global memory → total reads = $2 \times 32 \times 32 \times 32 = 65{,}536$
- With 16×16 tiles: total reads drop to $\frac{65{,}536}{16} = 4{,}096$

**Challenge for Attention**: Tiling works directly for matmul, but attention includes a **row-wise softmax** that depends on the *entire* row. We cannot compute softmax for a tile until we have seen all tiles in that row.

→ This is solved by **online softmax**.

In [ ]:
# TODO: Implement tiled matrix multiplication
# Demonstrate tiled matmul vs naive matmul (pure Python for clarity)
def naive_matmul(A, B):
    """Element-wise matmul — each C[i,j] reads a full row/col from global memory."""
    M, K_ = A.shape
    K2, N = B.shape
    C = torch.zeros(M, N, device=A.device)
    global_reads = 0
    for i in range(M):
        for j in range(N):
            for k in range(K_):
                C[i, j] += A[i, k] * B[k, j]
                global_reads += 2  # read A[i,k] and B[k,j]
    return C, global_reads

"""
Real HIP Kernel Pseudocode:
    ```hip
    __global__ void tiledMatMul(float* A, float* B, float* C, int M, int N, int K) {
        __shared__ float As[TILE_SIZE][TILE_SIZE];
        __shared__ float Bs[TILE_SIZE][TILE_SIZE];

        int tx = threadIdx.x, ty = threadIdx.y;
        int row = blockIdx.y * TILE_SIZE + ty;
        int col = blockIdx.x * TILE_SIZE + tx;

        float sum = 0.0f;
        for (int t = 0; t < (K + TILE_SIZE - 1) / TILE_SIZE; t++) {
            // Load tiles from global to shared memory
            int tiledCol = t * TILE_SIZE + tx;
            int tiledRow = t * TILE_SIZE + ty;
            As[ty][tx] = (row < M && tiledCol < K) ? A[row * K + tiledCol] : 0.0f;
            Bs[ty][tx] = (tiledRow < K && col < N) ? B[tiledRow * N + col] : 0.0f;
            __syncthreads();  // Wait for all threads to load

            // Compute partial dot product
            for (int k = 0; k < TILE_SIZE; k++) {
                sum += As[ty][k] * Bs[k][tx];
            }
            __syncthreads();  // Wait for all threads to finish
        }

        // Write result to global memory
        if (row < M && col < N) {
            C[row * N + col] = sum;
        }
    }
    ```
"""
def tiled_matmul(A, B, tile_size=4):
    """Tile-based matmul — tiles are loaded into 'shared memory' once.

    Args:
        A: (M, K) input matrix
        B: (K, N) input matrix
        tile_size: size of each tile block

    Returns:
        C: (M, N) output matrix
        global_reads: number of global memory reads
    """
    M, K_ = A.shape
    K2, N = B.shape
    C = torch.zeros(M, N, device=A.device)
    global_reads = 0

    # TODO: Loop over tiles of C (ti, tj)
    for ti in range(0, M, tile_size):
        for tj in range(0, N, tile_size):
            # C_tile accumulates in "registers"
            # TODO: Loop over tiles of K (tk)
            for tk in range(0, K_, tile_size):
                # TODO: Load A-tile and B-tile into "shared memory"
                # Each tile is read once from global memory
                A_tile = A[None, None]  # Replace with correct code
                B_tile = B[None, None]  # Replace with correct code

                # Each tile load reads tile_size*tile_size elements
                global_reads += 2 * tile_size * tile_size  # Replace with correct code

                # TODO: Accumulate tile multiplication into C
                C[ti:ti+tile_size, tj:tj+tile_size] += None # Replace with correct code

    return C, global_reads

# Compare on an 8×8 matrix
size = 8
A = torch.randn(size, size)
B = torch.randn(size, size)

C_naive, reads_naive = naive_matmul(A, B)
C_tiled, reads_tiled = tiled_matmul(A, B, tile_size=4)

print(f"Naive  global reads: {reads_naive:,}")
print(f"Tiled  global reads: {reads_tiled:,}   (tile_size=4)")
print(f"Reduction: {reads_naive / reads_tiled:.1f}×")
print(f"Results match: {torch.allclose(C_naive, C_tiled, atol=1e-5)}")

Naive  global reads: 1,024
Tiled  global reads: 256   (tile_size=4)
Reduction: 4.0×
Results match: True


## 4. Online Softmax — The Key Enabler

### What is Softmax?

The **softmax** function converts a vector of arbitrary real numbers into a probability distribution. For an input vector $x = [x_1, x_2, \dots, x_N]$:

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_{j=1}^{N} e^{x_j}}$$

In attention mechanisms, softmax is applied to each row of the $N \times N$ attention score matrix, converting scores into attention weights that sum to 1.

### Numerical Stability: The Max Subtraction Trick

Direct computation of $e^{x_i}$ can overflow for large $x_i$. The standard fix is to subtract the maximum value:

$$\text{softmax}(x)_i = \frac{e^{x_i - m}}{\sum_{j=1}^{N} e^{x_j - m}}, \quad \text{where } m = \max_j x_j$$

This is mathematically equivalent (dividing numerator and denominator by $e^m$) but ensures the largest exponent is $e^0 = 1$.

### Standard Softmax: Two-Pass Algorithm

Computing softmax requires **two passes** over the data:

```
Algorithm: Standard Softmax
Input: x = [x₁, x₂, ..., xₙ]
Output: p = [p₁, p₂, ..., pₙ] where pᵢ = softmax(x)ᵢ

Pass 1 (REDUCE - Find statistics):
  Step 1: m = max(x)           # Find maximum for numerical stability
  Step 2: s = Σ exp(xⱼ - m)    # Compute sum of exponentials

Pass 2 (MAP - Normalize):
  Step 3: For each i: pᵢ = exp(xᵢ - m) / s
```

### Resource Analysis: Memory-Bound vs. Compute-Bound

| Step | Operation | Type | Resource Required | HBM Access |
|------|-----------|------|-------------------|------------|
| **Pass 1, Step 1** | `m = max(x)` | **REDUCE** | Memory-bound | Read all $N$ elements |
| **Pass 1, Step 2** | `s = Σexp(xⱼ - m)` | **REDUCE** | Memory-bound | Read all $N$ elements |
| **Pass 2, Step 3** | `pᵢ = exp(xᵢ - m) / s` | **MAP** | Compute-bound | Read $x$, write $p$ ($2N$ elements) |

**Key Observations:**

1. **REDUCE operations (Steps 1-2) are memory-bound**: They must read the entire input from HBM. The computation (finding max, summing) is trivial compared to the memory access cost.

2. **MAP operation (Step 3) is compute-bound**: Each element can be processed independently with exponential and division operations.

3. **Total HBM traffic for standard softmax**: $O(N)$ reads + $O(N)$ writes = **$O(N)$ per row**, or **$O(N^2)$ for the full attention matrix**.

### The Problem with Standard Softmax in Attention

For attention with sequence length $N$:
- The score matrix $S = QK^T$ has shape $N \times N$
- Standard softmax must **materialize the full $N \times N$ matrix** in HBM
- This requires $O(N^2)$ HBM storage and bandwidth
- For $N=4096$, this is **33.6 MB per head** (FP16) — too large for SRAM!

### Online Softmax: Single-Pass Solution

**Online softmax** solves this by maintaining **running statistics** that can be updated incrementally as new data arrives:

| Property | Standard Softmax | Online Softmax |
|----------|-----------------|----------------|
| **Passes** | 2 passes (must see all data first) | 1 pass (process as stream) |
| **State** | Must store full input | Only stores $(m, d)$ — 2 values! |
| **HBM Traffic** | $O(N)$ per row | $O(N)$ per row, but **tiles fit in SRAM** |
| **Use Case** | Small vectors | Large matrices (attention) |

### Online Softmax Algorithm

Maintain running maximum $m$ and running sum $d$:

```
Algorithm: Online Softmax
Input: Stream of blocks [b₁, b₂, ..., bₖ]
Output: Softmax probabilities

Initialize: m = -∞, d = 0

For each block b:
  1. m_block = max(b)                    # Find block maximum
  2. m_new = max(m, m_block)             # Update running max
  3. d = d × exp(m - m_new)              # Rescale old sum
     + Σ exp(bⱼ - m_new)                 # Add block contribution
  4. m = m_new                           # Update running max

Final: p = exp(x - m) / d                # Normalize all elements
```

### Update Formulas

$$m_{\text{new}} = \max(m_{\text{old}}, m_{\text{block}})$$
$$d_{\text{new}} = d_{\text{old}} \cdot e^{m_{\text{old}} - m_{\text{new}}} + \sum_{j \in \text{block}} e^{x_j - m_{\text{new}}}$$

**Why the rescaling?** When we discover a new maximum $m_{\text{new}} > m_{\text{old}}$, we must adjust the old sum:
$$d_{\text{old}} \cdot e^{m_{\text{old}} - m_{\text{new}}} = \sum_{\text{old}} e^{x - m_{\text{new}}}$$

### Why Online Softmax Enables FlashAttention

| Aspect | Standard Softmax | Online Softmax |
|--------|-----------------|----------------|
| **Data access** | Must load entire row | Can process tile-by-tile |
| **Intermediate storage** | $O(N^2)$ for full matrix | $O(N)$ for running stats |
| **SRAM usage** | Cannot fit $N \times N$ matrix | Running stats fit in registers |
| **HBM traffic** | $O(N^2)$ | $O(N^2 d^2 / M)$ with tiling |

By processing attention scores **tile by tile** and keeping running statistics $(m, d)$ in SRAM, FlashAttention avoids materializing the $N \times N$ matrix entirely. This is the key enabler for memory-efficient attention!

In [ ]:
# TODO: Implement online softmax
def online_softmax(x: torch.Tensor, block_size: int = 4) -> torch.Tensor:
    """Compute softmax of 1-D tensor x using online (incremental) algorithm.

    Processes x in blocks of `block_size`, maintaining running max and sum.

    Algorithm:
    1. Initialize running max m = -inf and running sum d = 0
    2. For each block:
       - Find block max m_block
       - Update running max: m_new = max(m, m_block)
       - Rescale old sum: d = d * exp(m - m_new)
       - Add block contribution: d += sum(exp(block - m_new))
    3. Return: exp(x - m) / d
    """
    N = x.shape[0]

    # TODO: Initialize running statistics
    m =  torch.tensor(float("-inf"))   # running max, start with -inf
    d =  torch.tensor(0.0)  # running sum of exp, start with 0

    # TODO: Process each block
    for start in range(0, N, block_size):
        block = x[start : start + block_size]

        # TODO: Find block maximum
        # Hint: Use block.max() to find the maximum value in the current block
        m_block = None  # Replace with correct code

        # TODO: Update running maximum
        # Hint: m_new = max(m, m_block)
        m_new = None  # Replace with correct code

        # TODO: Re-scale old sum and add new block contribution
        # Hint: d_new = d_old * exp(m_old - m_new) + sum(exp(block - m_new))
        d = None  # Replace with correct code
        m = m_new

    # TODO: Final softmax values
    # Hint: exp(x - m) / d
    return None / d  # Replace with correct code


# Verify against PyTorch softmax
x = torch.randn(16)
result_online = online_softmax(x, block_size=4)
result_torch  = torch.softmax(x, dim=0)

print(f"Online softmax: {result_online[:6].tolist()}")
print(f"Torch softmax:  {result_torch[:6].tolist()}")
print(f"Max absolute error: {(result_online - result_torch).abs().max().item():.2e}")
print(f"Match: {torch.allclose(result_online, result_torch, atol=1e-6)}  ✓")

Online softmax: [0.02825923077762127, 0.0985398218035698, 0.012425200082361698, 0.011203604750335217, 0.012779663316905499, 0.08641627430915833]
Torch softmax:  [0.02825922891497612, 0.0985398218035698, 0.012425200082361698, 0.011203604750335217, 0.012779663316905499, 0.08641627430915833]
Max absolute error: 1.86e-09
Match: True  ✓


## 5. FlashAttention Algorithm

### FlashAttention V1 — Core Idea

Instead of computing the full $N \times N$ attention matrix, FlashAttention processes Q, K, V in **tiles**:

1. Divide K, V into blocks of $B_c$ rows, and Q into blocks of $B_r$ rows
2. For each Q-block, iterate over K/V-blocks:
   - Load Q-tile, K-tile, V-tile into shared memory (SRAM)
   - Compute local $S_{\text{tile}} = Q_{\text{tile}} K_{\text{tile}}^T$
   - Update running softmax statistics (online softmax)
   - Accumulate output: $O \mathrel{{+}{=}} \text{softmax\_tile} \times V_{\text{tile}}$
3. After all K/V-blocks are processed, apply final rescaling

**HBM access**: $O(N^2 d^2 / M)$ where $M$ is SRAM size — much less than $O(Nd + N^2)$.

### FlashAttention V2 — Improvements

1. **Swap inner/outer loop**: V1 loops K,V (outer) × Q (inner); V2 loops Q (outer) × K,V (inner), avoiding redundant writes to O
2. **Defer rescaling**: V1 rescales O every iteration; V2 only rescales once at the end
3. **Better warp partitioning**: V2 splits Q across warps (not K), eliminating inter-warp communication

### FlashAttention V1 vs V2: Code Comparison

Let's compare the two algorithms side by side to understand why V2 is faster.

#### FlashAttention V1: K/V-Outer Loop

```python
# V1 Pseudocode: K/V blocks on outside
for j in range(0, N, block_size):          # Outer: K/V blocks
    Kj, Vj = K[j:j+Bs], V[j:j+Bs]          # Load K, V tile
    for i in range(0, N, block_size):      # Inner: Q blocks
        Qi = Q[i:i+Br]                      # Load Q tile
        S_tile = Qi @ Kj.T                  # Compute scores
        m_tile = S_tile.max(dim=-1)         # Block max
        m_new = max(m[i:i+Br], m_tile)      # Update running max
        correction = exp(m[i:i+Br] - m_new) # Rescale factor
        O[i:i+Br] *= correction             # RESCALE EVERY ITERATION!
        O[i:i+Br] += exp(S_tile - m_new) @ Vj
        m[i:i+Br] = m_new
```

**V1 Issues:**
1. **Rescales O every iteration** — writes to O repeatedly ($O(N^2)$ writes total)
2. **K/V on outside** — each SM processes different K/V blocks, poor data reuse
3. **Inter-warp communication** — Q split across warps requires synchronization

#### FlashAttention V2: Q-Outer Loop

```python
# V2 Pseudocode: Q blocks on outside
for i in range(0, N, block_size):          # Outer: Q blocks
    Qi = Q[i:i+Br]                          # Load Q tile (stays in SRAM)
    m[i:i+Br] = -inf                        # Init per-row stats
    l[i:i+Br] = 0
    O[i:i+Br] = 0
    for j in range(0, N, block_size):      # Inner: K/V blocks
        Kj, Vj = K[j:j+Bs], V[j:j+Bs]      # Load K, V tile
        S_tile = Qi @ Kj.T                  # Compute scores
        m_tile = S_tile.max(dim=-1)
        m_new = max(m[i:i+Br], m_tile)
        correction = exp(m[i:i+Br] - m_new)
        # DEFER RESCALING: accumulate unnormalized
        O[i:i+Br] += exp(S_tile - m_new) @ Vj
        l[i:i+Br] = l[i:i+Br] * correction + exp(S_tile - m_new).sum()
        m[i:i+Br] = m_new
    O[i:i+Br] /= l[i:i+Br]                  # FINAL RESCALE ONLY
```

**V2 Advantages:**
1. **Defer rescaling** — O is written once per Q-block, not per K/V-block ($O(N)$ writes vs $O(N^2)$)
2. **Q on outside** — each SM owns a Q-block and processes all K/V against it (better data locality)
3. **No inter-warp communication** — each warp handles complete Q-blocks

#### Why V2 is Faster: Summary

| Aspect | V1 | V2 | Impact |
|--------|-----|-----|--------|
| Rescaling | Every K/V iteration | Once at end | **Fewer non-matmul ops** |
| O writes | $O(N^2)$ | $O(N)$ | **Less HBM traffic** |
| Loop order | K/V-outer | Q-outer | **Better SM utilization** |
| Warp partitioning | K-split | Q-split | **No inter-warp sync** |

In [ ]:
# TODO: Implement FlashAttention V2 simulation
def flash_attention_sim(Q, K, V, block_size=4):
    """Simplified FlashAttention V2 (Python simulation).

    This mirrors the real algorithm's logic but runs on CPU/GPU tensors
    without custom CUDA kernels. The purpose is educational.

    Args:
        Q, K, V: (seq_len, head_dim)
        block_size: tile size for K/V blocks
    Returns:
        O: (seq_len, head_dim)
    """
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    # TODO: Initialize output and running statistics
    O = None  # Output accumulator
    l = None  # Running sum of exp (denominator)
    m = None  # Running max (for numerical stability)

    # TODO: Outer loop over Q blocks (V2 style)
    for j in range(0, N, block_size):
        # TODO: Load K and V blocks into "SRAM"
        Kj = None  # Replace with correct code
        Vj = None  # Replace with correct code

        # TODO: Compute local attention scores for ALL query positions vs. this K-block
        S_block = None  # Replace with correct code

        # TODO: Online softmax update
        m_block = None  # Block maximum
        m_new   = None  # Updated running max

        # TODO: Correction factor for previously accumulated values
        correction = None  # exp(m - m_new)

        # TODO: New exponentials for this block
        p_block = None  # exp(S_block - m_new)

        # TODO: Update running statistics
        l_new = None  # l * correction + p_block.sum()

        # TODO: Update output accumulator
        # Hint: rescale old O and add new contribution
        O = None  # Replace with correct code

        m = m_new
        l = l_new

    # TODO: Final normalization
    O = None  # Replace with correct code
    return O


# Verify against standard attention
N, d = 16, 8
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

out_standard = standard_attention(
    Q.unsqueeze(0).unsqueeze(0),
    K.unsqueeze(0).unsqueeze(0),
    V.unsqueeze(0).unsqueeze(0),
).squeeze()

out_flash = flash_attention_sim(Q, K, V, block_size=4)

print(f"Standard output shape: {out_standard.shape}")
print(f"Flash sim output shape: {out_flash.shape}")
print(f"Max error: {(out_standard - out_flash).abs().max().item():.2e}")
print(f"Results match: {torch.allclose(out_standard, out_flash, atol=1e-5)}  ✓")

Standard output shape: torch.Size([16, 8])
Flash sim output shape: torch.Size([16, 8])
Max error: 1.49e-07
Results match: True  ✓


## 6. Benchmarking: Standard vs. Flash Attention

PyTorch ≥ 2.0 provides `torch.nn.functional.scaled_dot_product_attention` (SDPA) which automatically selects the most efficient backend: **FlashAttention**, **Memory-Efficient Attention**, or the **Math** fallback.

We compare wall-clock time and peak memory for increasing sequence lengths.

In [8]:
def benchmark_attention(seq_lengths, batch=1, heads=8, head_dim=64, warmup=3, repeats=10):
    """Benchmark standard vs SDPA (FlashAttention backend) attention."""
    results = []

    for N in seq_lengths:
        Q = torch.randn(batch, heads, N, head_dim, device=device, dtype=torch.float16)
        K = torch.randn(batch, heads, N, head_dim, device=device, dtype=torch.float16)
        V = torch.randn(batch, heads, N, head_dim, device=device, dtype=torch.float16)

        # --- Standard attention ---
        torch.cuda.synchronize() if device.type == "cuda" else None
        for _ in range(warmup):
            _ = standard_attention(Q, K, V)
        torch.cuda.synchronize() if device.type == "cuda" else None

        torch.cuda.reset_peak_memory_stats() if device.type == "cuda" else None
        t0 = time.perf_counter()
        for _ in range(repeats):
            _ = standard_attention(Q, K, V)
        torch.cuda.synchronize() if device.type == "cuda" else None
        std_time = (time.perf_counter() - t0) / repeats * 1000  # ms
        std_mem = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else 0

        # --- SDPA (FlashAttention backend) ---
        for _ in range(warmup):
            _ = F.scaled_dot_product_attention(Q, K, V)
        torch.cuda.synchronize() if device.type == "cuda" else None

        torch.cuda.reset_peak_memory_stats() if device.type == "cuda" else None
        t0 = time.perf_counter()
        for _ in range(repeats):
            _ = F.scaled_dot_product_attention(Q, K, V)
        torch.cuda.synchronize() if device.type == "cuda" else None
        sdpa_time = (time.perf_counter() - t0) / repeats * 1000
        sdpa_mem = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else 0

        speedup = std_time / sdpa_time if sdpa_time > 0 else float("inf")
        results.append((N, std_time, sdpa_time, speedup, std_mem, sdpa_mem))

        print(f"N={N:>5d} | Standard: {std_time:7.2f} ms, {std_mem:8.1f} MB | "
              f"SDPA: {sdpa_time:7.2f} ms, {sdpa_mem:8.1f} MB | "
              f"Speedup: {speedup:.2f}×")

    return results


if device.type == "cuda":
    print("=== Attention Benchmark ===")
    seq_lengths = [256, 512, 1024, 2048, 4096]
    results = benchmark_attention(seq_lengths)
else:
    print("GPU not available — benchmark skipped.")
    print("The SDPA function still works on CPU but without FlashAttention speedup.")
    # Quick correctness check on CPU
    N, H, D = 32, 2, 16
    Q = torch.randn(1, H, N, D)
    K = torch.randn(1, H, N, D)
    V = torch.randn(1, H, N, D)
    out_std  = standard_attention(Q, K, V)
    out_sdpa = F.scaled_dot_product_attention(Q, K, V)
    print(f"CPU correctness check: match={torch.allclose(out_std, out_sdpa, atol=1e-5)}")

=== Attention Benchmark ===
N=  256 | Standard:    0.07 ms,     36.7 MB | SDPA:    0.06 ms,     34.6 MB | Speedup: 1.07×
N=  512 | Standard:    0.09 ms,     44.0 MB | SDPA:    0.13 ms,     35.7 MB | Speedup: 0.68×
N= 1024 | Standard:    0.28 ms,     71.3 MB | SDPA:    0.24 ms,     37.8 MB | Speedup: 1.18×
N= 2048 | Standard:    0.61 ms,    176.2 MB | SDPA:    0.40 ms,     42.0 MB | Speedup: 1.55×
N= 4096 | Standard:    2.30 ms,    587.2 MB | SDPA:    1.40 ms,     50.5 MB | Speedup: 1.64×


## 7. IO Complexity Comparison

| Method | HBM Reads/Writes | Peak HBM for intermediates |
|--------|----------------:|:--------------------------:|
| Standard Attention | $O(Nd + N^2)$ | $O(N^2)$ (stores full score matrix) |
| FlashAttention V1 | $O(N^2 d^2 / M)$ | $O(N)$ (only stores row-wise stats) |
| FlashAttention V2 | Same asymptotic, fewer non-matmul ops, better warp utilization | $O(N)$ |

Where $M$ = SRAM size per SM (typically 100–200 KB), $N$ = sequence length, $d$ = head dimension.

For typical settings ($d=64{-}128$, $M \gg d^2$), FlashAttention eliminates almost all $N^2$-sized HBM traffic.

In [ ]:
# TODO: Complete the memory analysis function
# Calculate theoretical memory savings
def memory_analysis(N, d, dtype_bytes=2):
    """Compare memory footprint of standard vs flash attention.

    Args:
        N: sequence length
        d: head dimension
        dtype_bytes: bytes per element (2 for FP16)

    Returns:
        std_intermediate: memory for standard attention intermediates
        flash_intermediate: memory for flash attention intermediates
    """
    # TODO: Standard attention stores S (N×N) and P (N×N)
    std_intermediate = None  # Replace with correct formula

    # TODO: Flash attention only stores running max, sum, and output per row
    # Hint: m (1 per row), l (1 per row), O (d per row)
    flash_intermediate = None  # Replace with correct formula

    return std_intermediate, flash_intermediate

print("=== Intermediate Memory Comparison (per head, FP16) ===")
print(f"{'N':>6s} | {'Standard':>12s} | {'Flash':>12s} | {'Savings':>8s}")
print("-" * 48)
for N in [512, 1024, 2048, 4096, 8192, 16384]:
    std_mem, flash_mem = memory_analysis(N, d=128)
    print(f"{N:>6d} | {std_mem/1e6:>10.2f} MB | {flash_mem/1e6:>10.4f} MB | "
          f"{std_mem/flash_mem:>6.0f}×")

=== Intermediate Memory Comparison (per head, FP16) ===
     N |     Standard |        Flash |  Savings
------------------------------------------------
   512 |       1.05 MB |     0.1331 MB |      8×
  1024 |       4.19 MB |     0.2662 MB |     16×
  2048 |      16.78 MB |     0.5325 MB |     32×
  4096 |      67.11 MB |     1.0650 MB |     63×
  8192 |     268.44 MB |     2.1299 MB |    126×
 16384 |    1073.74 MB |     4.2598 MB |    252×


## Conclusions

### Technical Concepts Learned
- **GPU Memory Hierarchy**: Registers → Shared Memory (SRAM) → L2 → HBM (DRAM); bandwidth differs by 10–100×
- **Tiling**: Partitioning matrices into tiles that fit in SRAM to reduce global memory round-trips
- **Online Softmax**: Incrementally computing softmax with running max and sum, enabling tile-by-tile attention
- **FlashAttention V1**: Tiled attention with online softmax, reducing HBM traffic from $O(N^2)$ to $O(N^2 d^2 / M)$
- **FlashAttention V2**: Improved loop ordering (Q-outer), deferred rescaling, better warp partitioning
- **SDPA**: PyTorch's `scaled_dot_product_attention` automatically uses FlashAttention when available

### Experiment Further
- Install `flash-attn` package and use `flash_attn.flash_attn_func` directly
- Compare FlashAttention performance across FP16 and BF16
- Benchmark with causal masks (autoregressive) vs. bidirectional attention
- Profile a Transformer block with `torch.profiler` to see real SRAM/HBM usage

---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
